In [2]:
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

2026-10-29 14:15:25.139628: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-10-29 14:15:25.537988: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-10-29 14:15:29.511633: I tensorflow/c/logging.cc:34] Successfully opened dynamic library libdirectml.d6f03b303ac3c4f2eeb8ca631688c9757b361310.so
2026-10-29 14:15:29.511821: I tensorflow/c/logging.cc:34] Successfully opened dynamic library libdxcore.so
2026-10-29 14:15:29.520394: I tensorflow/c/logging.cc:34] Successfully opened dynamic library libd

In [3]:
IMG_SIZE = (160, 160)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    "dataset/train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=True
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    "dataset/validation",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False
)

Found 20000 files belonging to 2 classes.


2026-10-29 14:15:32.594267: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-10-29 14:15:32.595549: I tensorflow/c/logging.cc:34] DirectML: creating device on adapter 0 (Intel(R) Iris(R) Xe Graphics)
2026-10-29 14:15:32.656137: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-10-29 14:15:32.656197: W tensorflow/core/common_runtime/pluggable_device/pluggable_device_bfc_allocator.cc:28] Overriding allow_growth setting because force_memory_growth was requested by the device.
2026-10-29 14:15:32.656222: I tensorflow/core/common_runtime/pluggable_device/plug

Found 5000 files belonging to 2 classes.


In [ ]:
print(train_ds.class_names)

AttributeError: 'MapDataset' object has no attribute 'class_names'

In [4]:
# Apply MobileNetV2's required input preprocessing.
train_ds = train_ds.map(
    lambda images, labels: (preprocess_input(images), labels)
)
val_ds = val_ds.map(
    lambda images, labels: (preprocess_input(images), labels)
)

In [5]:
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(160, 160, 3)
)

In [6]:
# Freeze the pretrained feature extractor.
base_model.trainable = False

inputs = layers.Input(shape=(160, 160, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = Model(inputs, outputs)

In [7]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 160, 160, 3)]     0         
                                                                 


 mobilenetv2_1.00_160 (Funct  (None, 5, 5, 1280)       2257984   
 ional)                                                          
                                                                 
 global_average_pooling2d (G  (None, 1280)             0         
 lobalAveragePooling2D)                                          
                                                                 
 dropout (Dropout)           (None, 1280)              0         
                                                                 
 dense (Dense)               (None, 1)                 1281      
                                                                 
Total params: 2,259,265
Trainable params: 1,281
Non-trainable params: 2,257,984
_________________________________________________________________


In [8]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=2,
        restore_best_weights=True
    )
]

In [9]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=callbacks
)

Epoch 1/5


2026-10-29 14:15:36.924212: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
2026-10-29 14:15:37.265598: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-10-29 14:15:37.265667: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 6941 MB memory) -> physical PluggableDevice (device: 0, name: DML, pci bus id: <undefined>)


625/625 [==============================] - ETA: 0s - loss: 0.0866 - accuracy: 0.9677

2026-10-29 14:18:01.447915: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
2026-10-29 14:18:01.728959: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-10-29 14:18:01.729044: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 6941 MB memory) -> physical PluggableDevice (device: 0, name: DML, pci bus id: <undefined>)


625/625 [==============================] - 189s 268ms/step - loss: 0.0866 - accuracy: 0.9677 - val_loss: 0.0462 - val_accuracy: 0.9852
Epoch 2/5
625/625 [==============================] - 145s 232ms/step - loss: 0.0486 - accuracy: 0.9815 - val_loss: 0.0440 - val_accuracy: 0.9868
Epoch 3/5
625/625 [==============================] - 144s 231ms/step - loss: 0.0440 - accuracy: 0.9836 - val_loss: 0.0449 - val_accuracy: 0.9864
Epoch 4/5
625/625 [==============================] - 163s 261ms/step - loss: 0.0404 - accuracy: 0.9846 - val_loss: 0.0472 - val_accuracy: 0.9848


In [10]:
val_loss, val_accuracy = model.evaluate(val_ds)
print("Validation accuracy:", val_accuracy)

157/157 [==============================] - 28s 177ms/step - loss: 0.0440 - accuracy: 0.9868
Validation accuracy: 0.9868000745773315


In [11]:
import numpy as np

for images, labels in val_ds.take(1):
    probabilities = model.predict(images, verbose=0)
    print("Predicted probabilities:", probabilities[:5].flatten())
    print("True labels:", labels[:5].numpy().flatten())
    break

2026-10-29 14:26:46.441965: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


Predicted probabilities: [5.7119742e-04 3.4725043e-04 8.6312546e-05 2.3922858e-05 7.1080698e-04]
True labels: [0. 0. 0. 0. 0.]


-------------------------------------------------
Can stop here if Feature extrction is enough or proceed if fine tuning is needed
-------------------------------------------------

In [13]:
base_model.trainable = True

# Freeze all but the last 20 layers.
for layer in base_model.layers[:-20]:
    layer.trainable = False

# Keep Batch Normalization layers frozen for stability.
for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

In [14]:
# Recompile with a much smaller learning rate.
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

fine_tune_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=3,
    callbacks=callbacks
)

Epoch 1/3


2026-10-29 14:27:21.179350: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
2026-10-29 14:27:21.561366: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-10-29 14:27:21.561460: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 6941 MB memory) -> physical PluggableDevice (device: 0, name: DML, pci bus id: <undefined>)


625/625 [==============================] - ETA: 0s - loss: 0.0422 - accuracy: 0.9840

2026-10-29 14:29:38.598369: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.
2026-10-29 14:29:38.921618: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-10-29 14:29:38.921708: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 6941 MB memory) -> physical PluggableDevice (device: 0, name: DML, pci bus id: <undefined>)


625/625 [==============================] - 169s 264ms/step - loss: 0.0422 - accuracy: 0.9840 - val_loss: 0.0405 - val_accuracy: 0.9872
Epoch 2/3
625/625 [==============================] - 173s 276ms/step - loss: 0.0306 - accuracy: 0.9889 - val_loss: 0.0393 - val_accuracy: 0.9874
Epoch 3/3
625/625 [==============================] - 181s 290ms/step - loss: 0.0229 - accuracy: 0.9920 - val_loss: 0.0412 - val_accuracy: 0.9852


In [15]:
val_loss, val_accuracy = model.evaluate(val_ds)
print("Validation accuracy:", val_accuracy)

157/157 [==============================] - 30s 189ms/step - loss: 0.0412 - accuracy: 0.9852
Validation accuracy: 0.985200047492981


In [16]:
import numpy as np

for images, labels in val_ds.take(1):
    probabilities = model.predict(images, verbose=0)
    print("Predicted probabilities:", probabilities[:5].flatten())
    print("True labels:", labels[:5].numpy().flatten())
    break

2026-10-29 14:39:29.014077: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


Predicted probabilities: [8.16215179e-05 2.86513168e-05 2.70846863e-06 5.76374987e-07
 1.00583486e-04]
True labels: [0. 0. 0. 0. 0.]


----------------------

In [17]:
import os

# Replace 'dataset/train' with your actual training folder path
data_dir = 'dataset/train'  
class_names = sorted(os.listdir(data_dir))

print(class_names)


['cats', 'dogs']


In [18]:
import numpy as np

for images, labels in val_ds.take(1):
    probabilities = model.predict(images, verbose=0)
    
    # Round numbers less than 0.5 to 0, and greater than 0.5 to 1
    predicted_classes = np.round(probabilities).flatten().astype(int)
    
    true_classes = labels.numpy().flatten().astype(int)
    
    print("Predicted Prediction Numbers:", predicted_classes[:5])
    print("True Prediction Numbers:     ", true_classes[:5])
    break


Predicted Prediction Numbers: [0 0 0 0 0]
True Prediction Numbers:      [0 0 0 0 0]
